SEC fillings

1. Companies are required to file many financial reports with the SEC each year.
2. An Important form is the 10-K, which is an annual report of the companies activities.
3. These forms are public records, and can be accessed through the SEC's EDGAR database.

Data Cleaning

1. Completed 10-k forms are available to download as text files  that contain XML components
2. In order to work with them, the following cleaning steps were applied.
    1. Cleaned up the files using regex
    2. Parsed XML into python data structures using Beautiful soup
    3. Extracted CIK (Central Index Key) ID which is a company identifier used by SEC
    4. Extracted specific sections of the form (items 1, 1a,7 and 7a)
3. You can look in the data directory in the notebook if you did like to examine the cleaned data for yourself.


Plan of attack

1. Split form sections into chunks using a Langchain text splitter.
2. Create a graph where each chunk is a node, adding chunk metadata as properties.
3. Create a vector index.
4. Calculate the text embedding vector for each chunk and populate the index.
5. Use Similarity search to find relevant chunks


In [1]:
import os
from dotenv import load_dotenv

import json
import textwrap

# Langchain

from langchain_community.graphs import Neo4jGraph
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Neo4jVector
from langchain.chains import RetrievalQAWithSourcesChain
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings


In [ ]:
load_dotenv()

#Environment variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

#GLobal Constants
VECTOR_INDEX_NAME = 'form_10k_chunks'  # single 10k documents
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'text'
VECTOR_EMBEDDING_PROPERTY = 'textEmbedding'


In [3]:
first_file_name = "/Users/apple/Desktop/GenAI/GenAI/Knowledge_Graphs/Data/0000950170-23-027948.json"

In [4]:
first_file_as_object = json.load(open(first_file_name))

In [5]:
type(first_file_as_object)

dict

In [6]:
#will check for keys and type of values of object in dictionary


for k,v in first_file_as_object.items():
    print(k,type(v))

item1 <class 'str'>
item1a <class 'str'>
item7 <class 'str'>
item7a <class 'str'>
cik <class 'str'>
cusip6 <class 'str'>
cusip <class 'list'>
names <class 'list'>
source <class 'str'>


In [8]:
item1_text = first_file_as_object['item1']

In [10]:
item1_text[0:1500]

'>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved cloud’, provi

As we can see the text is large and hence we will perform chunking so that we are not going to take entire text and store that in a single record.

chunking is done using textsplitter from langchain.

In [13]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 200,
    length_function = len,
    is_separator_regex = False,
)

In [14]:
item1_text_chunks = text_splitter.split_text(item1_text)

In [15]:
type(item1_text_chunks)

list

In [16]:
len(item1_text_chunks)

254

In [17]:
item1_text_chunks[0]

'>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved cloud’, provi

In [21]:
# will create an helper function which will go through file and each section and create a chunk out of them.

def split_form10k_data_from_file(file):
    chunks_with_metadata = []
    file_as_object = json.load(open(file))
    
    for item in['item1','item1a','item7','item7a']:
        print(f'Processing {item} from {file}')
        item_text = file_as_object[item]
        item_text_chunks = text_splitter.split_text(item_text)
        chunk_seq_id = 0
        
        for chunk in item_text_chunks[:20]:
            form_id = file[file.rindex('/') + 1:file.rindex('.')]
            
            chunks_with_metadata.append({
                'text':chunk,
                'f10kItem': item,
                'chunkSeqId': chunk_seq_id,
                'formId': f'{form_id}',
                'chunkId': f'{form_id}-{item}-chunk{chunk_seq_id:04d}',
                'names': file_as_object['names'],
                'cik': file_as_object['cik'],
                'cusip6': file_as_object['cusip6'],
                'source': file_as_object['source'],
            })
            chunk_seq_id +=1
        print(f'\tSplit into {chunk_seq_id} chunks')
    return chunks_with_metadata

In [19]:
first_file_chunks = split_form10k_data_from_file(first_file_name)

Processing item1 from /Users/apple/Desktop/GenAI/GenAI/Knowledge_Graphs/Data/0000950170-23-027948.json


NameError: name 'form_id' is not defined